# Collision Avoidance - Train Model (ResNet18)

Welcome to this host side Jupyter Notebook!  This should look familiar if you ran through the notebooks that run on the robot.  In this notebook we'll train our image classifier to detect two classes
``free`` and ``blocked``, which we'll use for avoiding collisions.  For this, we'll use a popular deep learning library *PyTorch*

In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.datasets as datasets
import torchvision.models as models
import torchvision.transforms as transforms

### Upload and extract dataset

Before you start, you should upload the ``dataset.zip`` file that you created in the ``data_collection.ipynb`` notebook on the robot to Google Drive

Run the code below to mount Google Drive on the runtime using the authorization code.
Since it says Go to this URL in a browser: http: // ..., first click this link to allow access.
After permission copy this code, switch to the application and paste. A string is displayed with the indication, enter it in Enter your authorization code: and press the return key.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls '/content/drive/My Drive'



In [ ]:
!ls /content/data/

collision_avoidance_dataset.zip


Google Drive is mounted in /content/drive/My Drive.

Copy the data.zip file under /content/jetbot/ and unzip it.

If you uploaded to something_folder_name in Google Drive, replace it with /content/drive/MyDrive/something_folder_name/dataset.zip.


In [ ]:
!cp '/content/data/collision_avoidance_dataset.zip' '/content/data/test'
!cd /content/data/test
!rm -rf dataset/
!unzip /content/data/test/collision_avoidance_dataset.zip

Archive:  /content/data/test/collision_avoidance_dataset.zip
   creating: jetbot/notebooks/collision_avoidance/dataset/
   creating: jetbot/notebooks/collision_avoidance/dataset/blocked/
  inflating: jetbot/notebooks/collision_avoidance/dataset/blocked/036f69de-d7fc-11f1-9973-48b02d2e8c18.jpg  
  inflating: jetbot/notebooks/collision_avoidance/dataset/blocked/022c6860-d7fc-11f1-9973-48b02d2e8c18.jpg  
  inflating: jetbot/notebooks/collision_avoidance/dataset/blocked/b4994460-d7fb-11f1-9973-48b02d2e8c18.jpg  
  inflating: jetbot/notebooks/collision_avoidance/dataset/blocked/2dc26228-d7fb-11f1-9973-48b02d2e8c18.jpg  
  inflating: jetbot/notebooks/collision_avoidance/dataset/blocked/2c387212-d7fb-11f1-9973-48b02d2e8c18.jpg  
  inflating: jetbot/notebooks/collision_avoidance/dataset/blocked/10ff01b8-d7fc-11f1-9973-48b02d2e8c18.jpg  
  inflating: jetbot/notebooks/collision_avoidance/dataset/blocked/1206c082-d7fc-11f1-9973-48b02d2e8c18.jpg  
  inflating: jetbot/notebooks/collision_avoidance/

You should see a folder named ``dataset`` appear in the file browser.

### Create dataset instance

Now we use the ``ImageFolder`` dataset class available with the ``torchvision.datasets`` package.  We attach transforms from the ``torchvision.transforms`` package to prepare the data for training.

In [ ]:
dataset = datasets.ImageFolder(
    '/content/jetbot/notebooks/collision_avoidance/dataset',
    transforms.Compose([
        transforms.ColorJitter(0.1, 0.1, 0.1, 0.1),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
)

### Split dataset into train and test sets

Next, we split the dataset into *training* and *test* sets.  The test set will be used to verify the accuracy of the model we train.

In [ ]:
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [len(dataset) - 50, 50])

### Create data loaders to load data in batches

We'll create two ``DataLoader`` instances, which provide utilities for shuffling data, producing *batches* of images, and loading the samples in parallel with multiple workers.

In [ ]:
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
)

### Define the neural network

Now, we define the neural network we'll be training.  The *torchvision* package provides a collection of pre-trained models that we can use.

In a process called *transfer learning*, we can repurpose a pre-trained model (trained on millions of images) for a new task that has possibly much less data available.

Important features that were learned in the original training of the pre-trained model are re-usable for the new task.  We'll use the ``resnet18`` model.

In [ ]:
model = models.resnet18(pretrained=True)

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 207MB/s]


The ``resnet18`` model was originally trained for a dataset that had 1000 class labels, but our dataset only has two class labels!  We'll replace
the final layer with a new, untrained layer that has only two outputs.

In [ ]:
model.fc = torch.nn.Linear(512, 2)

Finally, we transfer our model for execution on the GPU

In [ ]:
device = torch.device('cuda')
model = model.to(device)

### Train the neural network

Using the code below we will train the neural network for 30 epochs, saving the best performing model after each epoch.

> An epoch is a full run through our data.

In [ ]:
NUM_EPOCHS = 30
BEST_MODEL_PATH = 'best_model_resnet18.pth'
best_accuracy = 0.0

optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

for epoch in range(NUM_EPOCHS):

    for images, labels in iter(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()
        optimizer.step()

    test_error_count = 0.0
    for images, labels in iter(test_loader):
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        test_error_count += float(torch.sum(torch.abs(labels - outputs.argmax(1))))

    test_accuracy = 1.0 - float(test_error_count) / float(len(test_dataset))
    print('%d: %f' % (epoch, test_accuracy))
    if test_accuracy > best_accuracy:
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        best_accuracy = test_accuracy

0: 0.800000
1: 0.760000
2: 0.800000
3: 0.860000
4: 0.860000
5: 0.880000
6: 0.800000
7: 0.740000
8: 0.840000
9: 0.880000
10: 0.860000
11: 0.840000
12: 0.920000
13: 0.920000
14: 0.780000
15: 0.760000
16: 0.780000
17: 0.820000
18: 0.860000
19: 0.820000
20: 0.880000
21: 0.740000
22: 0.780000
23: 0.920000
24: 0.880000
25: 0.940000
26: 1.000000
27: 0.920000
28: 0.940000
29: 0.940000


Once that is finished, you should see a file ``best_model_resnet18.pth`` in the Jupyter Lab file browser.  Select ``Right click`` -> ``Download`` to download the model to your workstation